# NB06 — Skenario A: Evaluasi HDBSCAN Baseline

Evaluasi clustering HDBSCAN (mcs=50, ms=5 dari NB05) menggunakan ground-truth
label hasil verifikasi manual (`face_labels_verified.csv`).

**Metrik yang dihitung:**
- Internal (tidak butuh label): Coverage, Noise Rate, Silhouette, DBCV, DBI
- Eksternal (butuh ground-truth): Purity, ARI, NMI

**Input:** `output_nb01/`, `output_nb05/`, `output_labeling/`  
**Output:** `output_nb06/skenario_a_results.pkl`, `output_nb06/skenario_a_metrics.csv`

## 1. Instalasi

In [ ]:
!pip install -q hdbscan scikit-learn pandas
# faiss-gpu sudah tersedia di Colab GPU runtime
try:
    import faiss
    print("faiss OK:", faiss.__version__)
except ImportError:
    !pip install -q faiss-gpu
    import faiss

## 2. Import & Mount Drive

In [ ]:
import os, pickle, time, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix,
)
from hdbscan.validity import validity_index

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Drive mounted.")

## 3. Konfigurasi Path

In [ ]:
BASE       = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings")
NB01_DIR   = BASE / "output_nb01"    # embeddings.npy
NB05_DIR   = BASE / "output_nb05"    # metadata_labeled.pkl, cluster_summary.pkl
LABEL_DIR  = BASE / "output_labeling"  # face_labels_verified.csv
OUTPUT_DIR = BASE / "output_nb06"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBSAMPLE   = 2000   # untuk Silhouette & DBCV (sama dengan NB03)
RANDOM_SEED = 42

print("Config OK.")
print(f"  NB01 : {NB01_DIR}")
print(f"  NB05 : {NB05_DIR}")
print(f"  Label: {LABEL_DIR}")
print(f"  Out  : {OUTPUT_DIR}")

## 4. Load Data

In [ ]:
embeddings = np.load(NB01_DIR / "embeddings.npy").astype(np.float32)
N, D = embeddings.shape

with open(NB05_DIR / "metadata_labeled.pkl", "rb") as f:
    meta = pickle.load(f)
assert len(meta) == N, f"Mismatch: {len(meta)} meta vs {N} embeddings"

with open(NB05_DIR / "cluster_summary.pkl", "rb") as f:
    cluster_summary = pickle.load(f)

df_gt = pd.read_csv(LABEL_DIR / "face_labels_verified.csv")
assert len(df_gt) == N, f"Mismatch ground-truth: {len(df_gt)} vs {N}"

labels = np.array([m["cluster_id"] for m in meta])   # -1 = noise HDBSCAN

print(f"Embeddings : {N:,} wajah, {D} dimensi")
print(f"Cluster summary params: {cluster_summary['params']}")
print(f"Ground-truth: {len(df_gt):,} baris, {df_gt['identity_id'].max()+1} identitas unik")
print(f"Status distribusi: {dict(df_gt['status'].value_counts())}")

## 5. Metrik Internal (tanpa ground-truth)

Coverage, Noise Rate, Silhouette (cosine via FAISS), DBCV, DBI.
Metodologi identik dengan NB03 untuk komparabilitas.

In [ ]:
t_start = time.time()

# ── Coverage & Noise Rate ──────────────────────────────────────────────────
n_clusters  = int(cluster_summary['n_clusters'])
n_noise     = int((labels == -1).sum())
n_clustered = int((labels >= 0).sum())
coverage    = n_clustered / N * 100
noise_pct   = n_noise / N * 100

cluster_sizes = Counter(labels[labels >= 0].tolist())
sizes = sorted(cluster_sizes.values(), reverse=True)

print(f"{'='*50}")
print(f"  COVERAGE & NOISE")
print(f"{'='*50}")
print(f"  Total wajah   : {N:,}")
print(f"  Ter-cluster   : {n_clustered:,} ({coverage:.2f}%)")
print(f"  Noise         : {n_noise:,} ({noise_pct:.2f}%)")
print(f"  Jumlah cluster: {n_clusters}")
print(f"  Ukuran cluster: min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.1f}")

# ── Subsample untuk Silhouette & DBCV ─────────────────────────────────────
rng      = np.random.default_rng(seed=RANDOM_SEED)
idx_pool = np.where(labels >= 0)[0]
n_sub    = min(SUBSAMPLE, len(idx_pool))
idx_sub  = np.sort(rng.choice(idx_pool, size=n_sub, replace=False))

emb_sub = embeddings[idx_sub].copy().astype(np.float32)
faiss.normalize_L2(emb_sub)   # L2-norm → cosine similarity via inner product

# ── Silhouette (cosine distance, precomputed via FAISS) ──────────────────
print(f"\nMenghitung Silhouette (n_sub={n_sub})...")
t0 = time.time()
try:
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(D))
    print("  FAISS GPU aktif.")
except Exception:
    index = faiss.IndexFlatIP(D)
    print("  FAISS CPU (GPU tidak tersedia).")

index.add(emb_sub)
sim_matrix, _ = index.search(emb_sub, n_sub)
dist_matrix   = np.clip(1.0 - sim_matrix.astype(np.float64), 0.0, 2.0)
np.fill_diagonal(dist_matrix, 0.0)

labels_sub = labels[idx_sub]
sil_score  = silhouette_score(dist_matrix, labels_sub, metric='precomputed')
print(f"  Silhouette = {sil_score:.4f}  ({time.time()-t0:.1f}s)")

# ── DBCV ─────────────────────────────────────────────────────────────────
print("Menghitung DBCV...")
t0 = time.time()
mask_valid   = labels_sub >= 0
dist_sub     = dist_matrix[np.ix_(mask_valid, mask_valid)]
labels_valid = labels_sub[mask_valid]

# Workaround bug hdbscan 0.8.41: drop cluster < 3 titik di subsample
label_counts   = Counter(labels_valid.tolist())
valid_clusters = {lbl for lbl, cnt in label_counts.items() if cnt >= 3}
keep           = np.isin(labels_valid, list(valid_clusters))
dist_sub_v     = dist_sub[np.ix_(keep, keep)]
labels_v       = labels_valid[keep]

# Remap ke integer kontigu (requirement validity_index)
unique_labels = np.unique(labels_v)
remap         = {old: new for new, old in enumerate(unique_labels)}
labels_v      = np.array([remap[l] for l in labels_v])

try:
    dbcv_score = validity_index(dist_sub_v, labels_v, metric='precomputed', d=D)
    print(f"  DBCV = {dbcv_score:.4f}  ({time.time()-t0:.1f}s)")
except AssertionError as e:
    dbcv_score = float('nan')
    print(f"  DBCV gagal (hdbscan bug): {e}")

# ── DBI ───────────────────────────────────────────────────────────────────
print("Menghitung DBI...")
t0 = time.time()
emb_sub_f64 = emb_sub[mask_valid].astype(np.float64)
dbi_score   = davies_bouldin_score(emb_sub_f64, labels_valid)
print(f"  DBI = {dbi_score:.4f}  ({time.time()-t0:.1f}s)")
print(f"\nTotal waktu metrik internal: {time.time()-t_start:.1f}s")

## 6. Metrik Eksternal (Purity, ARI, NMI)

Menggunakan `face_labels_verified.csv` sebagai ground-truth.
Hanya wajah dengan `identity_id >= 0` (terverifikasi, bukan noise/discard).

In [ ]:
# Filter: hanya wajah terverifikasi
df_eval = df_gt[df_gt["identity_id"] >= 0].copy()

y_true = df_eval["identity_id"].values     # ground-truth label (integer)
y_pred = df_eval["proxy_cluster"].values   # HDBSCAN cluster assignment

print(f"Wajah untuk evaluasi eksternal: {len(df_eval):,}")
print(f"Identitas unik (ground-truth) : {len(np.unique(y_true))}")
print(f"Cluster unik (HDBSCAN pred)   : {len(np.unique(y_pred))}")

# ── Purity ────────────────────────────────────────────────────────────────
# Untuk setiap cluster predicted, hitung fraksi kelas mayoritas
def purity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    return float(cm.max(axis=0).sum() / cm.sum())

purity = purity_score(y_true, y_pred)
print(f"\nPurity = {purity:.4f}")

# ── ARI & NMI ─────────────────────────────────────────────────────────────
ari = adjusted_rand_score(y_true, y_pred)
nmi = normalized_mutual_info_score(y_true, y_pred, average_method='arithmetic')

print(f"ARI    = {ari:.4f}")
print(f"NMI    = {nmi:.4f}")

## 7. Ringkasan Metrik

In [ ]:
print("=" * 58)
print("  SKENARIO A — HDBSCAN BASELINE")
print(f"  Dataset: DOKUMENTASI OUTDOOR 2025")
print(f"  Params : mcs={cluster_summary['params']['min_cluster_size']}, "
      f"ms={cluster_summary['params']['min_samples']}, "
      f"method={cluster_summary['params']['cluster_selection_method']}")
print("=" * 58)

rows = [
    ("Coverage",        f"{coverage:.2f}%"),
    ("Noise Rate",      f"{noise_pct:.2f}%"),
    ("Jumlah Cluster",  str(n_clusters)),
    ("Silhouette",      f"{sil_score:.4f}"),
    ("DBCV (primer)",   f"{dbcv_score:.4f}" if not np.isnan(dbcv_score) else "NaN"),
    ("DBI",             f"{dbi_score:.4f}"),
    ("── Ground-Truth Metrics ──", ""),
    ("Wajah Dievaluasi",f"{len(df_eval):,} / {N:,}"),
    ("Identitas Unik",  str(len(np.unique(y_true)))),
    ("Purity",          f"{purity:.4f}"),
    ("ARI",             f"{ari:.4f}"),
    ("NMI",             f"{nmi:.4f}"),
]
for label, val in rows:
    if val == "":
        print(f"  {label}")
    else:
        print(f"  {label:<22}: {val}")
print("=" * 58)

## 8. Simpan Hasil

In [ ]:
skenario_a = {
    "skenario"   : "A",
    "deskripsi"  : "HDBSCAN baseline, tanpa enhancement, mcs=50 ms=5",
    "dataset"    : {
        "n_total"       : N,
        "n_clustered"   : n_clustered,
        "n_noise"       : n_noise,
        "n_verified"    : int(len(df_eval)),
        "n_identities"  : int(len(np.unique(y_true))),
    },
    "params"     : cluster_summary["params"],
    "metrics_internal": {
        "coverage_pct"  : round(coverage, 4),
        "noise_pct"     : round(noise_pct, 4),
        "n_clusters"    : n_clusters,
        "silhouette"    : round(float(sil_score), 4),
        "dbcv"          : round(float(dbcv_score), 4) if not np.isnan(dbcv_score) else None,
        "dbi"           : round(float(dbi_score), 4),
    },
    "metrics_external": {
        "purity"        : round(purity, 4),
        "ari"           : round(ari, 4),
        "nmi"           : round(nmi, 4),
    },
}

# Pickle (untuk dibaca notebook berikutnya)
with open(OUTPUT_DIR / "skenario_a_results.pkl", "wb") as f:
    pickle.dump(skenario_a, f)

# CSV (untuk dibaca manusia)
flat = skenario_a["metrics_internal"] | skenario_a["metrics_external"]
flat["skenario"] = "A"
pd.DataFrame([flat]).to_csv(OUTPUT_DIR / "skenario_a_metrics.csv", index=False)

print("Tersimpan:")
print(f"  {OUTPUT_DIR}/skenario_a_results.pkl")
print(f"  {OUTPUT_DIR}/skenario_a_metrics.csv")